In [1]:
import pandas as pd

# Load the dataset using relative path
df = pd.read_csv("states_all_extended.csv")

# Preview first few rows
df.head()


,PRIMARY_KEY,STATE,YEAR,ENROLL,TOTAL_REVENUE,FEDERAL_REVENUE,STATE_REVENUE,LOCAL_REVENUE,TOTAL_EXPENDITURE,INSTRUCTION_EXPENDITURE,...,G08_HI_A_READING,G08_HI_A_MATHEMATICS,G08_AS_A_READING,G08_AS_A_MATHEMATICS,G08_AM_A_READING,G08_AM_A_MATHEMATICS,G08_HP_A_READING,G08_HP_A_MATHEMATICS,G08_TR_A_READING,G08_TR_A_MATHEMATICS
0,1992_ALABAMA,ALABAMA,1992,NaN,2678885.0,304177.0,1659028.0,715680.0,2653798.0,1481703.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1992_ALASKA,ALASKA,1992,NaN,1049591.0,106780.0,720711.0,222100.0,972488.0,498362.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1992_ARIZONA,ARIZONA,1992,NaN,3258079.0,297888.0,1369815.0,1590376.0,3401580.0,1435908.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1992_ARKANSAS,ARKANSAS,1992,NaN,1711959.0,178571.0,958785.0,574603.0,1743022.0,964323.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1992_CALIFORNIA,CALIFORNIA,1992,NaN,26260025.0,2072470.0,16546514.0,7641041.0,27138832.0,14358922.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
# Check shape (rows, columns)
df.shape

# Show column types
df.dtypes.head(15)


PRIMARY_KEY                      object
STATE                            object
YEAR                              int64
ENROLL                          float64
TOTAL_REVENUE                   float64
FEDERAL_REVENUE                 float64
STATE_REVENUE                   float64
LOCAL_REVENUE                   float64
TOTAL_EXPENDITURE               float64
INSTRUCTION_EXPENDITURE         float64
SUPPORT_SERVICES_EXPENDITURE    float64
OTHER_EXPENDITURE               float64
CAPITAL_OUTLAY_EXPENDITURE      float64
A_A_A                           float64
G01_A_A                         float64
dtype: object

In [3]:
# Sort columns by how many missing values they have
missing = df.isnull().sum().sort_values(ascending=False)
missing[missing > 0].head(20)  # Top 20 columns with missing data


G08_HP_A_MATHEMATICS    1702
G08_HP_A_READING        1701
G04_HP_A_MATHEMATICS    1700
G04_HP_A_READING        1699
G08_AM_A_MATHEMATICS    1655
G08_AM_A_READING        1654
G04_AM_A_MATHEMATICS    1652
G04_AM_A_READING        1651
G08_TR_A_READING        1574
G08_TR_A_MATHEMATICS    1570
G08_AS_A_READING        1562
G08_AS_A_MATHEMATICS    1558
G04_AS_A_READING        1551
G04_AS_A_MATHEMATICS    1547
G04_TR_A_MATHEMATICS    1532
G04_TR_A_READING        1532
G08_BL_A_MATHEMATICS    1494
G08_BL_A_READING        1493
G04_BL_A_READING        1489
G04_BL_A_MATHEMATICS    1486
dtype: int64

In [4]:
key_vars = [
    'YEAR', 'STATE',
    'ENROLL',
    'TOTAL_EXPENDITURE',
    'INSTRUCTION_EXPENDITURE',
    'AVG_MATH_4_SCORE', 'AVG_MATH_8_SCORE',
    'AVG_READING_4_SCORE', 'AVG_READING_8_SCORE',
    'G08_A_A_MATHEMATICS', 'G08_A_A_READING',
]

for col in key_vars:
    if col in df.columns:
        percent_missing = df[col].isnull().mean()
        print(f"{col}: {percent_missing:.1%} missing")


YEAR: 0.0% missing
STATE: 0.0% missing
ENROLL: 28.6% missing
TOTAL_EXPENDITURE: 25.7% missing
INSTRUCTION_EXPENDITURE: 25.7% missing
G08_A_A_MATHEMATICS: 64.9% missing
G08_A_A_READING: 67.2% missing


In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Reload data (if needed)
df = pd.read_csv("states_all_extended.csv")

# Filter for key columns
cols = ['STATE', 'YEAR', 'ENROLL', 'INSTRUCTION_EXPENDITURE', 'G08_A_A_MATHEMATICS']
df = df[cols]
df = df.dropna(subset=['ENROLL', 'INSTRUCTION_EXPENDITURE', 'G08_A_A_MATHEMATICS'])

# Compute per-student spending
df['spend_per_student'] = df['INSTRUCTION_EXPENDITURE'] / df['ENROLL']
df['log_spending'] = np.log(df['spend_per_student'])

# Ensure images folder exists
img_dir = Path("images")
img_dir.mkdir(exist_ok=True)

# === FIGURE 1: Line plot ===
avg_by_year = df.groupby('YEAR')['G08_A_A_MATHEMATICS'].mean()

plt.figure(figsize=(8, 5))
avg_by_year.plot(marker='o', color='steelblue')
plt.title("National Avg 8th Grade Math Score (NAEP, 2000–2015)", fontsize=13)
plt.xlabel("Year", fontsize=11)
plt.ylabel("Math Score", fontsize=11)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(img_dir / "fig1_math_trend.png", dpi=200)
plt.close()

# === FIGURE 2: Bar chart ===
latest_year = df['YEAR'].max()
top_states = df[df['YEAR'] == latest_year].copy()
top_states['STATE'] = top_states['STATE'].str.replace("_", " ")
top_states = top_states.sort_values('spend_per_student', ascending=False).head(10)

plt.figure(figsize=(10, 5))
sns.barplot(data=top_states, x='STATE', y='spend_per_student', palette='Blues_d')
plt.title(f"Top 10 States by Instruction Spending per Student ({latest_year})", fontsize=13)
plt.xlabel("State", fontsize=11)
plt.ylabel("Spending per Student ($)", fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.ticklabel_format(axis='y', style='plain')  # Avoid scientific notation
plt.tight_layout()
plt.savefig(img_dir / "fig2_top_spending_states.png", dpi=200)
plt.close()

# === FIGURE 3: Scatter plot ===
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x='log_spending', y='G08_A_A_MATHEMATICS', alpha=0.7)
sns.regplot(data=df, x='log_spending', y='G08_A_A_MATHEMATICS', scatter=False, color='red')
plt.title("Instruction Spending vs 8th Grade Math Score (State-Year Panel)", fontsize=13)
plt.xlabel("Log(Instruction Spending per Student)", fontsize=11)
plt.ylabel("8th Grade Math Score", fontsize=11)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(img_dir / "fig3_spending_vs_score.png", dpi=200)
plt.close()

print("✅ All updated figures saved in /images")


C:\Users\Deniz Bilen\AppData\Local\Temp\ipykernel_28472\3269226507.py:43: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=top_states, x='STATE', y='spend_per_student', palette='Blues_d')


✅ All updated figures saved in /images


In [7]:
import pandas as pd
import numpy as np

# === Load and clean data ===
df = pd.read_csv("states_all_extended.csv")

# Keep only needed columns
cols = ['STATE', 'YEAR', 'ENROLL', 'INSTRUCTION_EXPENDITURE', 'G08_A_A_MATHEMATICS']
df = df[cols].dropna()

# Create per-student instruction spending and log
df['spend_per_student'] = df['INSTRUCTION_EXPENDITURE'] / df['ENROLL']
df = df[df['spend_per_student'] > 0]
df['log_spending'] = np.log(df['spend_per_student'])

# Clean state names for modeling
df['STATE'] = df['STATE'].str.upper().str.replace(" ", "_")

# Set up panel index (for fixed effects models)
df = df.set_index(['STATE', 'YEAR']).sort_index()

# Preview the cleaned data
df.head()


ENROLL  INSTRUCTION_EXPENDITURE  G08_A_A_MATHEMATICS  \
STATE   YEAR                                                           
ALABAMA 2000  730184.0                2551713.0                264.0   
        2003  727900.0                2817111.0                262.0   
        2005  729342.0                3053380.0                262.0   
        2007  743273.0                3653466.0                266.0   
        2009  745668.0                3836398.0                269.0   

              spend_per_student  log_spending  
STATE   YEAR                                   
ALABAMA 2000           3.494616      1.251224  
        2003           3.870190      1.353303  
        2005           4.186486      1.431862  
        2007           4.915376      1.592368  
        2009           5.144914      1.638009

In [10]:
!pip install linearmodels


   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 11.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 2.7/2.7 MB 15.8 MB/s eta 0:00:00

   ---------------------------------------- 0/6 [setuptools-scm]
   ---------------------------------------- 0/6 [setuptools-scm]
   ------ --------------------------------- 1/6 [interface-meta]
   ------------- -------------------------- 2/6 [Cython]
   ------------- -------------------------- 2/6 [Cython]
   ------------- -------------------------- 2/6 [Cython]
   ------------- -------------------------- 2/6 [Cython]
   ------------- -------------------------- 2/6 [Cython]
   ------------- -------------------------- 2/6 [Cython]
   ------------- -------------------------- 2/6 [Cython]
   ------------- -------------------------- 2/6 [Cython]
   ------------- -------------------------- 2/6 [Cython]

In [11]:
from linearmodels.panel import PanelOLS


In [12]:
import statsmodels.api as sm
from linearmodels.panel import PanelOLS


# === Reuse df from previous cell ===
df_reg = df.copy()

# Dependent and independent variables
y = df_reg['G08_A_A_MATHEMATICS']
X = df_reg['log_spending']

# === Baseline OLS ===
X_ols = sm.add_constant(X)
ols_model = sm.OLS(y, X_ols).fit(cov_type='HC1')
print("=== Baseline OLS ===")
print(ols_model.summary())

# === Variation 1: State fixed effects ===
fe_model = PanelOLS.from_formula(
    'G08_A_A_MATHEMATICS ~ log_spending + EntityEffects',
    data=df_reg
).fit(cov_type='clustered', cluster_entity=True)
print("\n=== Fixed Effects (State Only) ===")
print(fe_model.summary)

# === Variation 2: State + Year fixed effects ===
fe2_model = PanelOLS.from_formula(
    'G08_A_A_MATHEMATICS ~ log_spending + EntityEffects + TimeEffects',
    data=df_reg
).fit(cov_type='clustered', cluster_entity=True)
print("\n=== Two-Way Fixed Effects (State + Year) ===")
print(fe2_model.summary)


=== Baseline OLS ===
                             OLS Regression Results                            
Dep. Variable:     G08_A_A_MATHEMATICS   R-squared:                       0.160
Model:                             OLS   Adj. R-squared:                  0.158
Method:                  Least Squares   F-statistic:                     51.77
Date:                 Fri, 03 Oct 2025   Prob (F-statistic):           3.16e-12
Time:                         09:48:49   Log-Likelihood:                -1407.1
No. Observations:                  397   AIC:                             2818.
Df Residuals:                      395   BIC:                             2826.
Df Model:                            1                                         
Covariance Type:                   HC1                                         
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
const          25